In [1]:
import numpy as np
import pandas as pd
import torch
from torch import nn

data = pd.read_csv('train.csv')

In [2]:
data.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [3]:
data = data.to_numpy()
# 0th col are the labels of the images
# 1st col up to 784th col are pixel values

In [4]:
data

array([[1, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       ...,
       [7, 0, 0, ..., 0, 0, 0],
       [6, 0, 0, ..., 0, 0, 0],
       [9, 0, 0, ..., 0, 0, 0]], shape=(42000, 785))

In [5]:
labels = data[:, 0]
data = data[:, 1:785]

In [6]:
data.shape

(42000, 784)

In [7]:
train_split = int(0.8 * len(data))
y_train, y_test = labels[:train_split], labels[train_split:]
X_train, X_test = data[:train_split], data[train_split:]

In [9]:
X_train, X_test = torch.tensor(X_train, dtype=torch.float32), torch.tensor(X_test, dtype=torch.float32)
y_train, y_test = torch.tensor(y_train, dtype=torch.int64), torch.tensor(y_test, dtype=torch.int64)

In [10]:
class NumClassificationModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(784, 10)
        self.layer2 = nn.Linear(10, 10)
        self.layer3 = nn.Linear(10,10)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        x = self.relu(x)
        x = self.layer3(x)
        return x

In [11]:
model_1 = NumClassificationModel()

In [12]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params = model_1.parameters(), lr = 0.1)

In [13]:
X_train.shape

torch.Size([33600, 784])

In [15]:
y_logits = model_1(X_train)
print(y_logits, y_logits.shape)
y_pred = nn.Softmax(dim=1)(y_logits)
print(y_pred, y_pred.shape)
y_pred_labels = torch.argmax(y_pred, dim=1)
print(y_pred_labels, y_pred_labels.shape)

tensor([[ -2.3975,  -0.7379,   2.1996,  ...,   1.6786,  -1.8867,   2.4327],
        [ -5.5714,  -3.8829,  -6.1522,  ...,  -4.7394,   4.0444,   5.1866],
        [ -2.0475,   0.8663,  -0.3028,  ...,   0.9262,   2.7807,   1.6057],
        ...,
        [ -5.8524,  -3.2374,  -6.2404,  ...,  -1.1574,   7.4424,   2.2761],
        [-11.7864,  -1.5847,  -0.9524,  ...,   4.5975,   6.8768,   5.3736],
        [ -6.2991,  -3.2862,  -7.4634,  ...,  -3.7039,   7.2236,   3.9803]],
       grad_fn=<AddmmBackward0>) torch.Size([33600, 10])
tensor([[2.8267e-04, 1.4862e-03, 2.8039e-02,  ..., 1.6654e-02, 4.7113e-04,
         3.5403e-02],
        [3.5021e-06, 1.8949e-05, 1.9593e-06,  ..., 8.0472e-06, 5.2531e-02,
         1.6461e-01],
        [2.4341e-03, 4.4852e-02, 1.3932e-02,  ..., 4.7620e-02, 3.0420e-01,
         9.3950e-02],
        ...,
        [1.3099e-06, 1.7903e-05, 8.8866e-07,  ..., 1.4331e-04, 7.7827e-01,
         4.4404e-03],
        [2.7457e-10, 7.3990e-06, 1.3925e-05,  ..., 3.5818e-03, 3.4991e-0

In [22]:
def accuracy(actual_labels, pred_labels):
    '''
    both actual_labels and pred_labels are vectors where each entry is an example
    '''
    return (pred_labels == actual_labels).sum().item() / len(actual_labels) * 100

In [23]:
# training loop
epochs = 1000 

for epoch in range(epochs):
    model_1.train()
    
    # forward pass
    y_logits = model_1(X_train)
    y_pred = nn.Softmax(dim=1)(y_logits)
    y_pred_labels = torch.argmax(y_pred, dim=1)

    # calculate the loss
    loss = loss_fn(y_logits, y_train)
    acc = accuracy(y_train, y_pred_labels)

    # optmizier.zero_grad()
    optimizer.zero_grad()

    # loss.backward()
    loss.backward()

    # optimizer.step()
    optimizer.step()

    # testing loop
    model_1.eval()
    with torch.inference_mode():
        # forward pass
        y_test_logits = model_1(X_test)
        y_test_pred = nn.Softmax(dim=1)(y_test_logits)
        y_test_pred_labels = torch.argmax(y_test_pred, dim=1)

        # calculate the loss
        test_loss = loss_fn(y_test_logits, y_test)
        test_acc = accuracy(y_test, y_test_pred_labels)

        if epoch % 20 == 0: 
            print(f'epoch: {epoch} | train loss: {loss:.5f} | train acc: {acc}% | test loss: {test_loss:.5f} | test acc: {test_acc}%')


epoch: 0 | train loss: 2.30126 | train acc: 11.113095238095239% | test loss: 2.30086 | test acc: 11.30952380952381%
epoch: 20 | train loss: 2.30126 | train acc: 11.113095238095239% | test loss: 2.30085 | test acc: 11.30952380952381%
epoch: 40 | train loss: 2.30126 | train acc: 11.113095238095239% | test loss: 2.30085 | test acc: 11.30952380952381%
epoch: 60 | train loss: 2.30126 | train acc: 11.113095238095239% | test loss: 2.30085 | test acc: 11.30952380952381%
epoch: 80 | train loss: 2.30125 | train acc: 11.113095238095239% | test loss: 2.30085 | test acc: 11.30952380952381%
epoch: 100 | train loss: 2.30125 | train acc: 11.113095238095239% | test loss: 2.30085 | test acc: 11.30952380952381%
epoch: 120 | train loss: 2.30125 | train acc: 11.113095238095239% | test loss: 2.30085 | test acc: 11.30952380952381%
epoch: 140 | train loss: 2.30125 | train acc: 11.113095238095239% | test loss: 2.30085 | test acc: 11.30952380952381%
epoch: 160 | train loss: 2.30125 | train acc: 11.1130952380952